In [141]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
words = open('names.txt', 'r').read().splitlines()
print(torch.__version__, len(words), words[:5])

text = ''.join(words)
print(text[:50])
print(len(text))
chars = sorted(set(text))
print(chars)
print(len(chars))
stoi = {s: i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
print(stoi)
print(stoi['a'], stoi['z'], stoi['.'])   # should print: 1 26 0
itos = {i+1: s for i, s in enumerate(chars)}
itos[0] = '.'
print(itos[1], itos[26], itos[0])   # should print: a z .

2.15.0 32033 ['emma', 'olivia', 'ava', 'isabella', 'sophia']
emmaoliviaavaisabellasophiacharlottemiaameliaharpe
196113
['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
26
{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26, '.': 0}
1 26 0
a z .


In [142]:
import random

block_size = 3

def build_dataset(words):
    X, Y = [], []
    for word in words:
        context = [0] * block_size
        for ch in word + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])



torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [143]:
C = torch.randn((27,2))

emb.shape

torch.Size([22655, 3, 2])

In [146]:
W1 = torch.randn((6,300))
b1 = torch.randn(300)


h.shape

lre = torch.linspace(-3, 0, 1000)
lrs = 10**lre

W2 = torch.randn((300,27))
b2 = torch.randn(27)

parameters = [C, W1, b1, W2, b2]

for p in parameters: 
    p.requires_grad = True

lri = []
lossi = []
stepi = []
for i in range (30000): 

    #minibatch construct
    ix = torch.randint(0,Xtr.shape[0],(32,))

    emb = C[Xtr[ix]]
    h = torch.tanh(emb.view(-1,6) @ W1 + b1)
    logits = h @ W2 + b2

    loss = F.cross_entropy(logits, Ytr[ix])
    
    

    for p in parameters:
        p.grad = None
    loss.backward()

    lr = 0.1
    for p in parameters:
        p.data += -lr * p.grad

    #lri.append(lr)
    stepi.append(i)
    lossi.append(loss.item())

print(loss.item())



3.2929863929748535


In [149]:
#training split, dev/validation split, test split
#80%, 10%, 10%

emb = C[Xdev]
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
logits = h @ W2 + b2
print('dev', F.cross_entropy(logits, Ydev).item())



dev 2.5022096633911133


In [150]:
emb = C[Xtr]
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
logits = h @ W2 + b2
print('dev', F.cross_entropy(logits, Ytr).item())

dev 2.5141563415527344
